In [ ]:
import numpy as np
import pandas as pd
import joblib
import sqlalchemy as db
import matplotlib.pyplot as plt

In [ ]:
#Load Models
hmm = joblib.load('../models/hmm_model.pkl')
pca = joblib.load('../models/pca.pkl')
scaler = joblib.load('../models/scaler.pkl')
meta_model = joblib.load('../models/meta_model.pkl')

n_states = hmm.n_components

In [ ]:
#load data
engine = db.create_engine('sqlite:///../data/raw/data.db')
df = get_all_data(engine)

In [ ]:
#prepare for hmm
X_scaled = scaler.transform(df)
X_pca = pca.transform(X_scaled)
state_probs = hmm.predict_proba(X_pca)
state_pred = hmm.predict(X_pca)

df['state'] = state_pred
df['confidence'] = state_probs.max(axis=1)
df['entropy'] = -np.sum(state_probs * np.log(state_probs + 1e-8), axis=1)

for i in range(n_states):
    df[f'prob_state_{i}'] = state_probs[:, i]

In [ ]:
#meta model datas
duration, count = [], 1
for i in range(len(df)):
    if i == 0:
        duration.append(1)
    elif df['state'].iloc[i] == df['state'].iloc[i - 1]:
        count += 1
        duration.append(count)
    else:
        count = 1
        duration.append(1)
df['state_duration'] = duration

ROLL = 5
df['confidence_roll'] = df['confidence'].rolling(ROLL).mean()
for i in range(n_states):
    df[f'prob_state_{i}_roll'] = df[f'prob_state_{i}'].rolling(ROLL).mean()

meta_features = ['confidence', 'entropy', 'state_duration', 'confidence_roll'] + \
                [f'prob_state_{i}_roll' for i in range(n_states)]

df = df.dropna(subset=meta_features)

final_states = []
for i in range(len(df)):
    if i == 0:
        final_states.append(df['state'].iloc[i])
        continue
    curr = df['state'].iloc[i]
    prev = final_states[-1]
    if curr != prev:
        X_t   = df.iloc[i][meta_features].values.reshape(1, -1)
        proba = meta_model.predict_proba(X_t)[0][1]
        final_states.append(curr if proba > 0.75 else prev)
    else:
        final_states.append(curr)

df['filtered_state'] = final_states

df['returns'] = df['spy_close'].pct_change()
df = df.dropna(subset=['returns'])